# Reinforcement Learning — Deep Q-Network (DQN) on CartPole

Part 2 of the RL series: scaling from a **Q-table** to a **Q-network**. A neural
network learns to balance a pole on a cart (CartPole-v1) using **Deep Q-Learning** —
the algorithm behind DeepMind's Atari breakthrough.

**Approach:**
1. Meet the environment (continuous states, discrete actions)
2. Random policy baseline
3. From Q-table to Q-network — function approximation
4. The two tricks that make DQN stable: **experience replay** + **target network**
5. Training, evaluation & a step-by-step episode replay

*Prerequisite: `ReinforcementLearning.ipynb` (tabular Q-learning).*

In [1]:
import random
from collections import deque, namedtuple

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim

sns.set_style("whitegrid")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("gymnasium version:", gym.__version__)
print("torch version:    ", torch.__version__)
print("device:           ", device)

gymnasium version: 1.4.0
torch version:     2.8.0+cu128
device:            cpu


## 1. The Environment — CartPole-v1

A pole is hinged to a cart on a frictionless track. The agent pushes the cart left or
right to keep the pole upright.

- **State:** 4 continuous values — cart position, cart velocity, pole angle, pole angular velocity
- **Actions:** 2 — push left (0) or push right (1)
- **Reward:** +1 every step the pole stays up → the return *is* the episode length (max 500)
- **Episode ends** when the pole falls past ±12°, the cart leaves the track (±2.4), or 500 steps pass

Unlike FrozenLake's 16 discrete states, the state space here is **continuous** — a Q-table
can't index it. That's the motivation for a Q-*network*.

In [2]:
env = gym.make("CartPole-v1")
env.reset(seed=SEED)

n_states = env.observation_space.shape[0]
n_actions = env.action_space.n
print(f"State dims: {n_states} (continuous), Actions: {n_actions}")
print("State = [cart position, cart velocity, pole angle, pole angular velocity]")
print("Sample state:", env.observation_space.sample().round(3))

State dims: 4 (continuous), Actions: 2
State = [cart position, cart velocity, pole angle, pole angular velocity]
Sample state: [ 2.557 -2.311  0.403 -1.359]


## 2. Baseline — Random Policy

How long does pure chance keep the pole up?

In [3]:
def run_episode(env, policy_fn, max_steps=500):
    """Run one episode; policy_fn(state_np) -> action. Returns total reward (= steps)."""
    state, _ = env.reset()
    total_reward = 0
    for _ in range(max_steps):
        state, reward, terminated, truncated, _ = env.step(policy_fn(state))
        total_reward += reward
        if terminated or truncated:
            break
    return total_reward

random_returns = [run_episode(env, lambda s: env.action_space.sample())
                  for _ in range(200)]
print(f"Random policy — mean episode length: {np.mean(random_returns):.1f} steps (of 500)")

Random policy — mean episode length: 21.6 steps (of 500)


## 3. From Q-Table to Q-Network

In tabular Q-learning we stored one number per (state, action) pair. With continuous
states we instead learn a function:

$$Q(s, a; \theta) \approx Q^*(s, a)$$

a small neural network that maps a state vector to one Q-value per action. The update
is the same Bellman target as before, now fitted by gradient descent:

$$y = r + \gamma \max_{a'} Q(s', a'; \theta^-) \qquad
\mathcal{L} = \big(y - Q(s, a; \theta)\big)^2$$

Two tricks make this stable:

- **Experience replay** — store transitions $(s, a, r, s')$ in a buffer and train on
  random mini-batches, breaking the correlation between consecutive samples.
- **Target network** — compute the target $y$ with a *frozen copy* of the network
  ($\theta^-$), synced periodically, so the target isn't chasing a moving goal.

In [4]:
class QNetwork(nn.Module):
    """MLP: state (4) -> hidden -> hidden -> Q-values (2)."""
    def __init__(self, n_states, n_actions, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_states, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x):
        return self.net(x)


Transition = namedtuple("Transition", ["state", "action", "reward", "next_state", "done"])

class ReplayBuffer:
    """Fixed-size memory of past transitions for random mini-batch sampling."""
    def __init__(self, capacity=10_000):
        self.memory = deque(maxlen=capacity)

    def push(self, *args):
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        batch = random.sample(self.memory, batch_size)
        return Transition(*zip(*batch))   # transpose list-of-tuples -> tuple-of-lists

    def __len__(self):
        return len(self.memory)

## 4. The DQN Agent

ε-greedy action selection on top of the Q-network, plus one gradient step per
environment step once the buffer has warmed up.

In [5]:
class DQNAgent:
    def __init__(self, n_states, n_actions, lr=1e-3, gamma=0.99,
                 batch_size=64, buffer_size=10_000, target_sync=500):
        self.n_actions = n_actions
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_sync = target_sync      # steps between target-network syncs

        self.policy_net = QNetwork(n_states, n_actions).to(device)
        self.target_net = QNetwork(n_states, n_actions).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()              # target net is never trained directly

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.buffer = ReplayBuffer(buffer_size)
        self.steps = 0

    def act(self, state, epsilon):
        """ε-greedy: explore with prob. ε, else take argmax_a Q(s, a)."""
        if random.random() < epsilon:
            return random.randrange(self.n_actions)
        with torch.no_grad():
            s = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            return int(self.policy_net(s).argmax(dim=1).item())

    def remember(self, *transition):
        self.buffer.push(*transition)

    def train_step(self):
        """One gradient step on a random mini-batch from the replay buffer."""
        if len(self.buffer) < self.batch_size:
            return
        batch = self.buffer.sample(self.batch_size)

        states      = torch.as_tensor(np.array(batch.state),      dtype=torch.float32, device=device)
        actions     = torch.as_tensor(np.array(batch.action),     dtype=torch.int64,   device=device).unsqueeze(1)
        rewards     = torch.as_tensor(np.array(batch.reward),     dtype=torch.float32, device=device)
        next_states = torch.as_tensor(np.array(batch.next_state), dtype=torch.float32, device=device)
        dones       = torch.as_tensor(np.array(batch.done),       dtype=torch.float32, device=device)

        # Q(s, a) for the actions actually taken
        q_values = self.policy_net(states).gather(1, actions).squeeze(1)

        # Bellman target from the frozen target network (0 on terminal states)
        with torch.no_grad():
            next_q = self.target_net(next_states).max(dim=1).values
            targets = rewards + self.gamma * next_q * (1 - dones)

        loss = nn.functional.smooth_l1_loss(q_values, targets)  # Huber loss
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        self.steps += 1
        if self.steps % self.target_sync == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

## 5. Training

Same ε-decay schedule as the tabular notebook: start fully random, anneal toward greedy.

In [6]:
def train_dqn(env, episodes=400, epsilon=1.0, min_epsilon=0.02, decay_rate=0.985):
    agent = DQNAgent(n_states, n_actions)
    returns_history, eps_history = [], []
    eps = epsilon

    for ep in range(episodes):
        state, _ = env.reset(seed=SEED + ep)
        done, total = False, 0
        while not done:
            action = agent.act(state, eps)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            agent.remember(state, action, reward, next_state, terminated)
            agent.train_step()
            state = next_state
            total += reward

        eps = max(min_epsilon, eps * decay_rate)
        returns_history.append(total)
        eps_history.append(eps)

        if (ep + 1) % 50 == 0:
            print(f"Episode {ep+1:>3}/{episodes} — avg return (last 50): "
                  f"{np.mean(returns_history[-50:]):6.1f}, ε = {eps:.3f}")
    return agent, returns_history, eps_history

agent, returns_history, eps_history = train_dqn(env)

Episode  50/400 — avg return (last 50):   28.5, ε = 0.470


Episode 100/400 — avg return (last 50):   77.1, ε = 0.221


Episode 150/400 — avg return (last 50):  105.0, ε = 0.104


Episode 200/400 — avg return (last 50):  150.9, ε = 0.049


Episode 250/400 — avg return (last 50):  120.5, ε = 0.023


Episode 300/400 — avg return (last 50):  178.4, ε = 0.020


Episode 350/400 — avg return (last 50):  234.3, ε = 0.020


Episode 400/400 — avg return (last 50):  437.4, ε = 0.020


## 6. Training Curves

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

window = 20
moving_avg = np.convolve(returns_history, np.ones(window) / window, mode="valid")
axes[0].plot(moving_avg)
axes[0].axhline(500, ls="--", c="gray", alpha=0.6, label="max (500)")
axes[0].set_title(f"Episode Return (moving average, window={window})")
axes[0].set_xlabel("Episode"); axes[0].set_ylabel("Return (= steps balanced)")
axes[0].legend()

axes[1].plot(eps_history)
axes[1].set_title("Exploration Rate (ε) Decay")
axes[1].set_xlabel("Episode"); axes[1].set_ylabel("ε")

plt.tight_layout()
plt.show()

<Figure size 1300x400 with 2 Axes>

## 7. Evaluation — Trained Agent vs. Random

In [8]:
def greedy_action(state):
    with torch.no_grad():
        s = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        return int(agent.policy_net(s).argmax(dim=1).item())

trained_returns = [run_episode(env, greedy_action) for _ in range(100)]

print(f"Random policy : {np.mean(random_returns):.1f} steps")
print(f"Trained DQN   : {np.mean(trained_returns):.1f} steps")

plt.figure(figsize=(6, 4))
plt.bar(["Random", "DQN (greedy)"],
        [np.mean(random_returns), np.mean(trained_returns)],
        color=["#d62728", "#2ca02c"])
plt.ylabel("Mean episode length over 100 episodes")
plt.title("CartPole-v1 — Random vs. Learned Policy")
plt.show()

Random policy : 21.6 steps
Trained DQN   : 412.5 steps


<Figure size 600x400 with 1 Axes>

## 8. Watch One Episode Step by Step

Pole angle (radians) at each step — 0 is perfectly upright. A trained policy keeps
the angle tightly around zero with small corrections, instead of letting it drift
until it falls.

In [9]:
# find a seed where the trained policy shows its full balancing skill
best, best_state = 0, None
for probe_seed in range(20):
    s, _ = env.reset(seed=probe_seed)
    length, done = 0, False
    while not done:
        s, r, terminated, truncated, _ = env.step(greedy_action(s))
        done = terminated or truncated
        length += 1
    if length > best:
        best, best_state = length, probe_seed

state, _ = env.reset(seed=best_state)
print(f"{'step':>4} | {'cart pos':>8} | {'pole angle':>10} | action")
print("-" * 45)
done, step, total = False, 0, 0
while not done:
    action = greedy_action(state)
    state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    step += 1
    total += reward
    if step <= 12 or done:                       # first steps + the ending
        print(f"{step:>4} | {state[0]:>8.3f} | {state[2]:>10.4f} | "
              f"{'left' if action == 0 else 'right'}")
    elif step == 13:
        print(" ... (balancing) ...")
print("-" * 45)
print(f"Episode finished after {int(total)} steps "
      f"({'pole fell' if total < 500 else 'max score reached'})")

step | cart pos | pole angle | action
---------------------------------------------
   1 |    0.013 |    -0.0469 | left
   2 |    0.009 |    -0.0423 | left
   3 |    0.001 |    -0.0321 | left
   4 |   -0.011 |    -0.0164 | left
   5 |   -0.027 |     0.0050 | left
   6 |   -0.047 |     0.0321 | left
   7 |   -0.071 |     0.0651 | right
   8 |   -0.091 |     0.0924 | right
   9 |   -0.107 |     0.1144 | right
  10 |   -0.119 |     0.1310 | left
  11 |   -0.135 |     0.1542 | right
  12 |   -0.148 |     0.1725 | right
 ... (balancing) ...
 500 |    1.054 |    -0.0022 | left
---------------------------------------------
Episode finished after 500 steps (max score reached)


## 9. Conclusion

- **Function approximation** replaces the Q-table with a neural network, letting
  Q-learning handle continuous state spaces like CartPole's.
- **Experience replay** and a **target network** are what make training with a neural
  net stable — without them, correlated samples and moving targets cause divergence.
- The random baseline survives ~20 steps; the trained DQN balances the pole for
  hundreds of steps, often hitting the 500-step cap.
- Next steps: **Double DQN** (decouple action selection from evaluation),
  **Dueling DQN** (separate value/advantage streams), and **Prioritized Replay**
  (sample the most surprising transitions more often).